# Quantitative Association Rule Mining of Typhoon Outcomes

In this notebook, we apply **quantitative association rule mining (ARM)** to identify recurring links between weather intensity, flood-control spending conditions, and severe disaster outcomes.

The workflow mirrors a report-style structure:

- Integrate typhoon impact and infrastructure context data at Region-Year level
- Discretize continuous variables into interpretable high/low indicators
- Mine frequent itemsets and association rules under transparent thresholds
- Focus interpretation on outcome-related consequents
- Visualize top rules using heatmap, bubble, and lift bar-chart views

Compared with hypothesis testing, ARM is exploratory. It surfaces high-influence co-occurrence patterns that can guide deeper statistical or domain validation.


### Import

Start by importing required libraries for data handling, rule mining, and visualization.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mlxtend.frequent_patterns import apriori, association_rules
from pathlib import Path

if (Path.cwd() / 'data').exists():
    data_dir = Path('data')
elif (Path.cwd().parent / 'data').exists():
    data_dir = Path('../data')
else:
    raise FileNotFoundError("Could not locate data directory from current working directory")

out_dir = data_dir / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)
print(f'Using data directory: {data_dir.resolve()}')
print(f'Export directory: {out_dir.resolve()}')

## Loading and Merging the Complete Datasets

This section builds the analysis-ready table (`df`) using the same Region-Year integration logic used in the project notebooks.

- `typhoon-info-infra-project.csv` provides meteorological and infrastructure finance context
- `cleaned_typhoon_impacts.csv` provides disaster outcomes
- Context is aggregated to Region-Year, then merged with outcomes using an inner join

The resulting table represents one regional disaster profile per row.


In [ ]:
# 1. LOAD DATA
info_raw = pd.read_csv(data_dir / 'merged' / 'typhoon-info-infra-project.csv')
impact_raw = pd.read_csv(data_dir / 'merged' / 'cleaned_typhoon_impacts.csv')

# 2. AGGREGATE INFO DATA
info_agg = info_raw.groupby(['Region', 'Year'], as_index=False).agg({
    'Max 24-hour Rainfall (mm)': 'max',
    'Peak Gust (10 mins sustained) (m/s)': 'max',
    'Cumulative_Budget_To_Date': 'sum',
    'Variance_Ratio_To_Date': 'mean'
})

# 3. MERGE
df = impact_raw.merge(info_agg, on=['Region', 'Year'], how='inner').copy()
print(f'Shape: {df.shape}')

### Data Structure Check

Inspect data types and non-null counts to confirm all variables required for discretization and rule mining are present.


In [ ]:
df.info()

### Initial Sample Preview

Review a sample of merged observations before feature binning.


In [ ]:
df.head(15)

## Discretization and Rule Generation

Association Rule Mining in this notebook uses binary transaction-style features. Continuous variables are transformed using median-based splits for balanced, data-driven thresholds.

**Design choices:**

- `Rain_High`: rainfall above median
- `Budget_Low`: cumulative budget below median
- `Inefficient_Spending`: variance ratio above median
- Outcomes:
  - `Outcome_High_Mortality`
  - `Outcome_High_Damage`

**Rule-mining thresholds:**

- `min_support=0.05` keeps rules present in at least 5% of transactions
- `lift > 1.1` keeps rules at least 10% stronger than random co-occurrence


In [ ]:
def discretize_data(data):
    df_bins = pd.DataFrame()
    df_bins['Rain_High'] = data['Max 24-hour Rainfall (mm)'] > data['Max 24-hour Rainfall (mm)'].median()
    df_bins['Budget_Low'] = data['Cumulative_Budget_To_Date'] < data['Cumulative_Budget_To_Date'].median()
    df_bins['Inefficient_Spending'] = data['Variance_Ratio_To_Date'] > data['Variance_Ratio_To_Date'].median()
    df_bins['Outcome_High_Mortality'] = data['Deaths'] > data['Deaths'].median()
    df_bins['Outcome_High_Damage'] = data['Damage to Infrastructure (PhP)'] > data['Damage to Infrastructure (PhP)'].median()
    return df_bins


df_mining = discretize_data(df)

# 5. FREQUENT ITEMSETS
frequent_itemsets = apriori(df_mining, min_support=0.05, use_colnames=True)

# 6. GENERATE ALL RULES
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.1)

# 7. FILTER OUTCOME RULES
target_rules = rules[
    rules['consequents'].apply(lambda x: any('Outcome' in item for item in x))
].sort_values('lift', ascending=False).copy()

# 8. TOP RULES FOR VISUALIZATION
top_rules = target_rules.head(15).copy()


def get_outcome_label(consequents):
    items = list(consequents)
    has_mort = any('Mortality' in i for i in items)
    has_dmg = any('Damage' in i for i in items)
    if has_mort and has_dmg:
        return 'Both outcomes'
    if has_mort:
        return 'Mortality'
    if has_dmg:
        return 'Damage'
    return 'Other'


def short_label(row):
    ant = ', '.join(sorted(row['antecedents']))
    ant = (ant
        .replace('Outcome_High_Mortality', 'Mortality')
        .replace('Outcome_High_Damage', 'Damage')
        .replace('Inefficient_Spending', 'Ineff.')
        .replace('Budget_Low', 'Budget_Low')
        .replace('Rain_High', 'Rain_High'))
    return ant[:42] + '...' if len(ant) > 42 else ant


top_rules['outcome_type'] = top_rules['consequents'].apply(get_outcome_label)
top_rules['label'] = top_rules.apply(short_label, axis=1)

print(f'Total transactions: {len(df_mining)}')
print(f'All rules found: {len(rules)}')
print(f'Outcome rules: {len(target_rules)}')
print(f'Top rules (plotted): {len(top_rules)}')
if not top_rules.empty:
    print(f"Lift range in top: {top_rules['lift'].min():.2f} - {top_rules['lift'].max():.2f}")

print('\nTop 15 rules by lift:')
print(top_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].round(3).to_string(index=False))

# EXPORTS MOVED TO data/outputs
target_rules.to_csv(out_dir / 'typhoon_association_rules_full.csv', index=False)
top_rules.to_csv(out_dir / 'typhoon_association_rules_top15.csv', index=False)
print('Saved:', out_dir / 'typhoon_association_rules_full.csv')
print('Saved:', out_dir / 'typhoon_association_rules_top15.csv')

### Interpretation Guide for Rule Metrics

- **Support:** how frequently a pattern appears in all transactions
- **Confidence:** conditional reliability of the rule
- **Lift:** strength above random expectation

This notebook prioritizes **lift** for ranking influence, while support and confidence are used to evaluate practical reliability.


## Visualization 1: Outcome-Rule Heatmap

This heatmap compares lift values across top antecedent-condition bundles and outcome consequents. It is useful for quickly identifying which risk combinations carry the strongest non-random influence.


In [ ]:
def plot_refined_heatmap(rules_df, save_path=None):
    if rules_df.empty:
        print('No rules to plot!')
        return

    plot_data = rules_df.copy()
    plot_data['consequent_label'] = plot_data['consequents'].apply(
        lambda x: ', '.join(list(x)).replace('Outcome_High_', '').replace('_', ' ')
    )

    pivot = plot_data.pivot(index='label', columns='consequent_label', values='lift')

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        pivot,
        annot=True,
        fmt='.2f',
        cmap='YlGnBu',
        cbar_kws={'label': 'Lift (Influence Score)'},
        linewidths=0.5,
    )

    plt.title('Statistical Influence: Infrastructure & Weather vs. Disaster Outcomes', fontsize=14, pad=20)
    plt.xlabel('Disaster Outcome (Consequent)', fontsize=12)
    plt.ylabel('Risk Conditions (Antecedents)', fontsize=12)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print('Saved plot:', save_path)
    plt.show()


plot_refined_heatmap(top_rules, save_path=out_dir / 'top_rules_heatmap.png')

## Visualization 2: Support-Confidence Bubble Map

This scatter plot shows top rules in quality space:

- x-axis: support
- y-axis: confidence
- bubble size: lift
- color: predicted outcome type

It helps identify high-influence rules that are both reliable and sufficiently prevalent.


In [ ]:
sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(13, 8))

palette = {
    'Mortality': '#3B8BD4',
    'Damage': '#D85A30',
    'Both outcomes': '#7F77DD',
    'Other': '#888780',
}

sns.scatterplot(
    data=top_rules,
    x='support',
    y='confidence',
    size='lift',
    hue='outcome_type',
    sizes=(150, 1200),
    alpha=0.75,
    palette=palette,
    ax=ax,
)

for _, row in top_rules.iterrows():
    ax.annotate(
        row['label'],
        xy=(row['support'], row['confidence']),
        xytext=(8, 4),
        textcoords='offset points',
        fontsize=8,
        color='#444',
        ha='left',
        va='bottom',
    )

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, linewidth=1)
ax.text(ax.get_xlim()[1], 0.505, '50% confidence', fontsize=8, color='gray', ha='right', va='bottom')

ax.set_title('Top association rules: what drives disaster outcomes?', fontsize=14, pad=16, loc='left')
ax.set_xlabel('Support (how often this scenario occurs in the data)', fontsize=11)
ax.set_ylabel('Confidence (probability of the outcome given the antecedent)', fontsize=11)

color_handles = [
    mpatches.Patch(color=v, label=k)
    for k, v in palette.items()
    if k in top_rules['outcome_type'].values
]
ax.legend(handles=color_handles, title='Predicted outcome', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)

plt.tight_layout()
plt.savefig(out_dir / 'top_rules_chart.png', dpi=150, bbox_inches='tight')
print('Saved plot:', out_dir / 'top_rules_chart.png')
plt.show()

### Diagnostic Summary Tables

These quick checks are useful for validating rule volume, top-rule ordering, and lift distribution before interpretation.


In [ ]:
print(f'Total rules (all consequents): {len(rules)}')
print(f'Target rules (outcome only): {len(target_rules)}')

In [ ]:
print(
    target_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
    .sort_values('lift', ascending=False)
    .head(15)
    .to_string()
)

In [ ]:
print(target_rules['lift'].describe().round(2))

## Visualization 3: Lift-Based Rule Ranking

This bar chart gives a direct ranking of the top rules by lift, with a baseline marker at 1.0 to indicate the no-effect threshold.


In [ ]:
sns.set_theme(style='whitegrid')
plt.figure(figsize=(12, 10))

palette = {
    'Mortality': '#3B8BD4',
    'Damage': '#D85A30',
    'Both outcomes': '#7F77DD',
}

plot_df = top_rules.sort_values('lift', ascending=False)

barplot = sns.barplot(
    data=plot_df,
    x='lift',
    y='label',
    hue='outcome_type',
    palette=palette,
    dodge=False,
)

plt.axvline(x=1.0, color='black', linestyle='--', alpha=0.6, label='Baseline (No Effect)')
plt.title('The Escalation of Risk: How Infrastructure Factors Multiply Typhoon Impact', fontsize=16, pad=20, loc='left')
plt.xlabel('Lift Score (Multiplier of Risk)', fontsize=12)
plt.ylabel('Risk Factors (Antecedents)', fontsize=12)

for p in barplot.patches:
    width = p.get_width()
    if width > 0:
        plt.text(width + 0.05, p.get_y() + p.get_height() / 2, f'{width:.2f}x', va='center', fontsize=10, fontweight='bold')

plt.legend(title='Predicted Outcome', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(out_dir / 'top_rules_lift_bar.png', dpi=150, bbox_inches='tight')
print('Saved plot:', out_dir / 'top_rules_lift_bar.png')
plt.show()

## Assumptions and Limitations

- **Association is not causation:** ARM identifies co-occurrence, not causal effects.
- **Threshold sensitivity:** Median discretization and support/lift thresholds influence discovered rules.
- **Information compression:** Binary binning simplifies interpretation but may hide gradients in severity.
- **Dataset dependence:** Rules can shift with added years, region composition changes, or reporting updates.


## Conclusion

The quantitative ARM pipeline identifies high-lift relationships between weather and financing conditions and severe typhoon outcomes. Exported rule tables and visual summaries in `data/outputs` provide a reproducible base for report integration, policy discussion, and comparison with fuzzy ARM results.
